# Macrophage states to Visium HD data with scANVI 

**Goal.** Starting from the spatial object that already carries the malignant cell states
(notebook 1b_ figure 2-3, `adata_sp_final_relabelled.h5ad`), label the macrophages with the macrophage states from
snRNA-seq myeloid atlas (CD36high, INHBAhigh, STAB1high). Tumour states and all other labels are kept.

**Approach.** Same as previous notebook, on macrophages only: reference = snRNA-seq macrophages of the three
states (tumour timepoints); query = Visium HD macrophages. Joint scVI (batch = dataset,
covariate = sample/section) → scANVI → predicted state + confidence; low-confidence cells → `Uncertain`.

**Output.** `adata_sp_final_relabelled_with_macs.h5ad`: `cell_label_final` with tumour states,
macrophage states and published labels (fixed colours below), plus `macrophage_predicted_label`,
`macrophage_prediction_score`, `X_scANVI_macrophage`. 

The tumour predictions of previous notebook (`predicted_label`, `prediction_score`, `X_scANVI`) are kept unchanged.

In [ ]:
# !pip install --quiet scanpy squidpy "scvi-tools[cuda]"
# %cd <project>/spatial_visiumHD

In [ ]:
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import scvi
import squidpy as sq
import torch

## Config

In [ ]:
# --- inputs / outputs ---
reference_path = "./../h5ad/myeloid/sub_Myeloid_raw_scvi_combine_annotated.h5ad" 

model_dir, figures_output_dir, tables_output_dir = "models", "figures/macrophage", "tables"

# --- transfer ---
prefix = "macrophage"                                       # names of prediction columns / models / files
label_key = "cell_state"                                    # reference state column
state_colors = {
    "Macrophage CD36high": "#9c9ede", 
    "Macrophage INHBAhigh": "#d33f6a", 
    "Macrophage STAB1high": "#99103d",
    "Uncertain": "#bdbdbd",
}
uncertain_label = "Uncertain"                               # same label as low-confidence tumour cells (ref to notebook)
reference_timepoint_col = "recist_v3"
reference_timepoints = ["1_treatment_naive", "3_on_treatment", "4_TKI_resistance"]
reference_sample_col = "sample"

query_label_col = "cell_label_final"                        # labels after notebook 
query_target_label = "Macrophage"
query_sample_col = "sample"                                 # tissue section, e.g. ER_pre
query_group_col = "response_group"                          # ER / SR / LR1 / LR2 (plots only)
library_key = "slide"
final_label_col = "cell_label_final"                        # updated in place (macrophages only)

seed = 0
for d in (model_dir, figures_output_dir, tables_output_dir):
    os.makedirs(d, exist_ok=True)
sc.settings.figdir = figures_output_dir
scvi.settings.seed = seed
pred_col, score_col, latent_key = f"{prefix}_predicted_label", f"{prefix}_prediction_score", f"X_scANVI_{prefix}"
states = [s for s in state_colors if s != uncertain_label]
print("GPU available:", torch.cuda.is_available(), "| scvi-tools", scvi.__version__)

## 1. Reference states from the snRNA-seq atlas

In [ ]:
# --- model (same settings for every transfer) ---
counts_layers = ["counts", "raw_counts"] # first layer found is used as raw counts
min_cells_per_gene = 10
min_cells_per_reference_sample = 10      # reference samples with fewer cells of these states are dropped
n_latent = 30
scvi_max_epochs = 200
scanvi_max_epochs = 50
train_models = True                      # False -> load saved models from model_dir
confidence_threshold = 0.5               # max class probability


def use_raw_counts(adata, name):
    for layer in counts_layers:
        if layer in adata.layers:
            adata.X = adata.layers[layer].copy()
            break
    else:
        print(f"{name}: no counts layer found, using X")
    vals = adata.X[:1000].data if hasattr(adata.X, "data") else np.asarray(adata.X[:1000]).ravel()
    assert np.allclose(vals, np.round(vals)) and vals.min() >= 0, f"{name}: X must contain raw counts"


adata_ref = sc.read_h5ad(reference_path) #sub_Myeloid_raw_scvi_combine_annotated.h5ad

adata_ref = adata_ref[adata_ref.obs[label_key].isin(states)
                      & adata_ref.obs[reference_timepoint_col].isin(reference_timepoints)].copy()

n_per_sample = adata_ref.obs[reference_sample_col].value_counts()
small = n_per_sample[n_per_sample < min_cells_per_reference_sample].index
if len(small):
    print(f"dropping {len(small)} reference samples with < {min_cells_per_reference_sample} cells: {list(small)}")
    adata_ref = adata_ref[~adata_ref.obs[reference_sample_col].isin(small)].copy()
use_raw_counts(adata_ref, "reference")
pd.crosstab(adata_ref.obs[label_key], adata_ref.obs[reference_timepoint_col], margins=True)

## 2. Query cells from the spatial object

The full object is kept (`adata_sp`); only the target cells (`adata_q`) are used for label transfer.

In [ ]:
# spatial object from previous notebook: malignant states already in cell_label_final
adata_sp = sc.read_h5ad("adata_sp_final_relabelled.h5ad") 
use_raw_counts(adata_sp, "spatial")
adata_q = adata_sp[adata_sp.obs[query_label_col].astype(str) == query_target_label].copy()
print(f"{adata_q.n_obs} '{query_target_label}' cells of {adata_sp.n_obs}")
pd.crosstab(adata_q.obs[query_sample_col], adata_q.obs[query_group_col])

## 3. Shared genes and joint object

Genes are intersected, filtered for coverage in each dataset, and intersected again.
`batch_sample` (dataset + sample/section) is the nuisance covariate within each dataset.

In [ ]:
genes = adata_ref.var_names.intersection(adata_q.var_names)
print(f"shared genes: {len(genes)}")
adata_ref, adata_q = adata_ref[:, genes].copy(), adata_q[:, genes].copy()
sc.pp.filter_genes(adata_ref, min_cells=min_cells_per_gene)
sc.pp.filter_genes(adata_q, min_cells=min_cells_per_gene)
genes = adata_ref.var_names.intersection(adata_q.var_names)
adata_ref, adata_q = adata_ref[:, genes].copy(), adata_q[:, genes].copy()
print(f"shared genes after coverage filter: {len(genes)}")

for a, name, sample_col, label in [(adata_ref, "reference", reference_sample_col, None),
                                   (adata_q, "spatial", query_sample_col, "Unknown")]:
    a.obs["dataset"] = name
    a.obs["batch_sample"] = name + "_" + a.obs[sample_col].astype(str)
    if label is not None:
        a.obs[label_key] = label

adata = sc.concat([adata_ref, adata_q], join="inner", label="source", keys=["reference", "spatial"], index_unique="-")
adata.obs = adata.obs[["dataset", "batch_sample", label_key]]
adata.obs[label_key] = adata.obs[label_key].astype("category")
adata.obs["dataset"].value_counts()

## 4. scVI → scANVI

- `batch_key="dataset"`: main correction, snRNA-seq vs Visium HD.
- `categorical_covariate_keys=["batch_sample"]`: sample/section effects within each dataset.
- scANVI is initialised from scVI with reference states as labels, query cells `Unknown`.

In [ ]:
scvi_path = os.path.join(model_dir, f"scvi_{prefix}")
scanvi_path = os.path.join(model_dir, f"scanvi_{prefix}")
scvi.model.SCVI.setup_anndata(adata, batch_key="dataset", labels_key=label_key,
                              categorical_covariate_keys=["batch_sample"])

if train_models:
    vae = scvi.model.SCVI(adata, n_latent=n_latent)
    vae.train(max_epochs=scvi_max_epochs, early_stopping=True)
    vae.save(scvi_path, overwrite=True)
    scanvi_model = scvi.model.SCANVI.from_scvi_model(vae, labels_key=label_key, unlabeled_category="Unknown")
    scanvi_model.train(max_epochs=scanvi_max_epochs, early_stopping=True)
    scanvi_model.save(scanvi_path, overwrite=True)
else:
    scanvi_model = scvi.model.SCANVI.load(scanvi_path, adata=adata)

## 5. Predictions and confidence

The reference confusion table is a sanity check only (training cells, so optimistic).

In [ ]:
adata.obsm["X_scANVI"] = scanvi_model.get_latent_representation()
probs = scanvi_model.predict(adata, soft=True)
probs.index = adata.obs_names
adata.obs["predicted_label"] = probs.idxmax(axis=1)
adata.obs["prediction_score"] = probs.max(axis=1)

is_q = adata.obs["dataset"] == "spatial"
ref = adata.obs[~is_q]
print(f"reference agreement (training cells): {(ref['predicted_label'] == ref[label_key].astype(str)).mean():.1%}")
pd.crosstab(ref[label_key], ref["predicted_label"], normalize="index").round(2).to_csv(
    f"{tables_output_dir}/{prefix}_reference_confusion.csv")

q = adata.obs[is_q].copy()
q.index = q.index.str.removesuffix("-spatial")
q["label"] = q["predicted_label"].where(q["prediction_score"] >= confidence_threshold, uncertain_label)
print(f"query cells below {confidence_threshold}: {(q['label'] == uncertain_label).sum()} / {len(q)}")
q["label"].value_counts()

## 6. Update the spatial object

Target cells receive their predicted state (or the uncertain label); all other cells keep their
current label in `final_label_col`. Predictions, scores and latent are stored under this
transfer's prefix, so earlier transfers are not overwritten. Existing colours are kept; states
get their fixed colours and any new label a colour from `other_palette`.

In [ ]:
assert q.index.isin(adata_sp.obs_names).all(), "predicted barcodes not found in the spatial object"
adata_sp.obs[pred_col] = q["label"].reindex(adata_sp.obs_names)
adata_sp.obs[score_col] = q["prediction_score"].reindex(adata_sp.obs_names)
latent = pd.DataFrame(adata.obsm["X_scANVI"][is_q.to_numpy()], index=q.index)
adata_sp.obsm[latent_key] = latent.reindex(adata_sp.obs_names).to_numpy()   # NaN for other cells

base_col = final_label_col if final_label_col in adata_sp.obs else query_label_col
old_colors = {}
if f"{final_label_col}_colors" in adata_sp.uns and final_label_col in adata_sp.obs:
    old_colors = dict(zip(adata_sp.obs[final_label_col].cat.categories, adata_sp.uns[f"{final_label_col}_colors"]))
final = adata_sp.obs[base_col].astype(str)
target = adata_sp.obs[pred_col].notna()
final[target] = adata_sp.obs.loc[target, pred_col]
adata_sp.obs[final_label_col] = final.astype("category")

# --- colours of the final spatial labels ---
final_colors = {
    "1_Epithelial baseline": "#8dd3c7", "2_Differentiated epithelial": "#bebada", "3_Devp_stem-like": "#80b1d3",
    "4_Inflammatory EMT": "#fccde5", "5_Neurod_EMT-like": "#bc80bd", "6_Proliferative": "#ffed6f",
    "Alveolar cell type 2": "#E16A86", "B cell": "#AA9000", "Basal cell": "#00AA5A", "Club cell": "#00A6CA",
    "Endothelial cell": "#B675E0", "Erythrocyte": "#0804F5", "Fibroblast": "#EF9708",
    "Follicular dendritic cell": "#3B99B1", "Goblet Ciliated cell": "#E093C3", "Granulocyte": "#005682",
    "Macrophage CD36high": "#9c9ede", "Macrophage INHBAhigh": "#d33f6a", "Macrophage STAB1high": "#99103d",
    "Plasmablast": "#FCC602", "Smooth muscle cell": "#4A6FE3", "T cell": "#4B201D", "Uncertain": "#bdbdbd",
}

other_palette = ["#E16A86", "#AA9000", "#00AA5A", "#00A6CA", "#B675E0", "#0804F5", "#EF9708", "#3B99B1",
                 "#E093C3", "#005682", "#FCC602", "#4A6FE3", "#4B201D", "#7F3C37", "#91B88B", "#5C5C5C"]

                 
cats = list(adata_sp.obs[final_label_col].cat.categories)
if final_colors:                                            # exact palette for every final label
    missing = [c for c in cats if c not in final_colors]
    assert not missing, f"labels without a colour in final_colors: {missing}"
    color_map = dict(final_colors)
else:                                                       # keep existing colours, add the rest
    color_map = {**old_colors, **state_colors}
    new = [c for c in cats if c not in color_map]
    used = set(color_map.values())
    free = [c for c in other_palette if c not in used] or other_palette
    color_map.update({c: free[i % len(free)] for i, c in enumerate(new)})
adata_sp.uns[f"{final_label_col}_colors"] = [color_map[c] for c in cats]
print("Category -> colour mapping:")
for c in cats:
    print(f"  {c:35s} {color_map[c]}")

adata_sp.write_h5ad("adata_sp_final_relabelled_with_macs.h5ad") #Final object! 

adata_sp.obs[final_label_col].value_counts()

## 7. QC and figures

In [ ]:
sc.pp.neighbors(adata, use_rep="X_scANVI", random_state=seed)
sc.tl.umap(adata, random_state=seed)
sc.pl.umap(adata, color=["dataset", label_key, "predicted_label"], wspace=0.4, save=f"_{prefix}_scanvi_label_transfer.pdf")

scored = adata_sp.obs[adata_sp.obs[score_col].notna()]
fig, ax = plt.subplots(figsize=(6, 3))
for s, grp in scored.groupby(query_sample_col, observed=True)[score_col]:
    ax.hist(grp, bins=np.linspace(0, 1, 41), histtype="step", label=s)
ax.axvline(confidence_threshold, ls="--", c="red")
ax.set_xlabel("scANVI prediction score")
ax.legend(fontsize=7, frameon=False)
plt.savefig(f"{figures_output_dir}/{prefix}_prediction_score_per_sample.pdf", bbox_inches="tight")
plt.show()

In [ ]:
sq.pl.spatial_scatter(adata_sp, color=final_label_col, library_key=library_key, img=True, size=12,
                      save=f"spatial_{final_label_col}_{prefix}.pdf")
sq.pl.spatial_scatter(adata_sp[adata_sp.obs[score_col].notna()], color=score_col, library_key=library_key,
                      img=False, size=15, save=f"spatial_{prefix}_prediction_score.pdf")


tumour_states = ["1_Epithelial baseline", "2_Differentiated epithelial", "3_Devp_stem-like",
                 "4_Inflammatory EMT", "5_Neurod_EMT-like"]
mac_states = list(state_colors)[:3]
spatial_subsets = {                                         # file suffix -> labels shown on the tissue
    "macrophage_states": mac_states,
    "macs_Tc_and_tumour_states": mac_states + ["T cell"] + tumour_states,
    "macs_and_tumour_states": mac_states + tumour_states,
}


for suffix, labels in spatial_subsets.items():
    sub = adata_sp[adata_sp.obs[final_label_col].isin(labels)].copy()
    sub.obs[final_label_col] = sub.obs[final_label_col].cat.remove_unused_categories()
    sub.uns[f"{final_label_col}_colors"] = [color_map[c] for c in sub.obs[final_label_col].cat.categories]
    sq.pl.spatial_scatter(sub, color=final_label_col, library_key=library_key, img=True, img_alpha=0.1,
                          size=18, save=f"spatial_{suffix}.pdf")

State composition per tissue section (confident predictions only), grouped by response group with
pre/post sections side by side.

In [ ]:
sample_order = ["ER_pre", "ER_post", "SR_pre", "SR_post", "LR1_pre", "LR1_post", "LR2_pre", "LR2_post"]

obs_t = adata_sp.obs[adata_sp.obs[final_label_col].isin(states)]
props = pd.crosstab(obs_t[query_sample_col], obs_t[final_label_col], normalize="index").mul(100)
props = props.reindex(columns=states, fill_value=0)
props = props.loc[[s for s in sample_order if s in props.index]]
n_cells = obs_t[query_sample_col].value_counts().reindex(props.index)
props.assign(n_cells=n_cells).round(2).to_csv(f"{tables_output_dir}/{prefix}_state_percent_per_sample.csv")

fig, ax = plt.subplots(figsize=(max(4, 0.9 * len(props)), 4))
props.plot.bar(stacked=True, ax=ax, width=0.75, edgecolor="none", color=[state_colors[s] for s in states])
groups = props.index.str.rsplit("_", n=1).str[0]           # ER_pre -> ER
starts = [i for i in range(len(groups)) if i == 0 or groups[i] != groups[i - 1]]
for i, start in enumerate(starts):
    end = starts[i + 1] if i + 1 < len(starts) else len(groups)
    ax.text((start + end - 1) / 2, 103, groups[start], ha="center", va="bottom", fontsize=11)
    if start > 0:
        ax.axvline(start - 0.5, color="grey", ls="--", lw=1)
ax.set_xticklabels([f"{s}\n(n={n})" for s, n in n_cells.items()], rotation=90)
ax.set_ylim(0, 108)
ax.set_ylabel(f"{query_target_label} (%)")
ax.set_xlabel("")
ax.legend(bbox_to_anchor=(1.01, 1), loc="upper left", frameon=False)
ax.spines[["top", "right"]].set_visible(False)
plt.savefig(f"{figures_output_dir}/{prefix}_state_composition_by_sample.pdf", dpi=300, bbox_inches="tight") #Figure  6
plt.show()
props.round(1)